# Ames Housing — Data Cleaning

Part 1 of the Ames Housing project. This notebook handles missing-value investigation and cleaning
on the raw `train.csv` dataset, producing a cleaned checkpoint (`train_cleaned.csv`) used by the
analysis notebook that follows [`02_eda.ipynb`](02_eda.ipynb).

## Initial Expectations

Before touching the data, I don't have strong hypotheses about the exact hierarchy of features
driving `SalePrice` — but based on general intuition, I'd expect land area, location, and number
of rooms to be among the strongest predictors. These are untested assumptions, revisited once
actual analysis begins in the next notebook.

## Cleaning Approach

Goal: identify columns with missing values or low information value, and decide — case by case —
whether to fill or drop them. No column is dropped without a specific reason backed by a check
against the data, not assumption alone.

Starting with missing values, then reviewing for low-variance/noise columns afterward.

In [2]:
import pandas as pd

df = pd.read_csv("../data/train.csv")

df["MSSubClass"] = df["MSSubClass"].astype(
    str
)  # Convert MSSubClass to string type because it is a categorical variable represented as numbers

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

numeric_cols = df.select_dtypes(include="number").columns.tolist()
object_cols = df.select_dtypes(include=["object", "str"]).columns.tolist()

missing


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

## What features do I Drop and Fill?
This is where I make a decision that affects my model significantly — deciding what columns stays, and whichones leave. No column will be dropped blindly without proper analysis.
I will first drop the ID column.

I will divide missing cols into `significant` and `insignificant`. 

- **Significant** in the sense that the percentage of missing rows in the col is relatively high.
- **Insignificant** in the sense that the percentage of missing rows in the col could easily be neglected

In [3]:
df.drop("Id", axis=1, inplace=True)  # Drop the 'Id' column as it is not useful for analysis

sig_missing = missing[missing > (0.1 * len(df))].sort_values(ascending=False)  # Significant missing values are those that are greater than 1% of the total number of rows in the dataset.

sig_numeric = sig_missing.index.intersection(numeric_cols)
sig_categorical = sig_missing.index.intersection(object_cols)

print(f"Significant missing values:\n{sig_missing}")

Significant missing values:
PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
FireplaceQu     690
LotFrontage     259
dtype: int64


In [4]:
insig_missing = missing[missing <= (0.1 * len(df))].sort_values(ascending=False)  # insignificant missing values are those that are less than or equal to 1% of the total number of rows in the dataset.

insig_numeric = insig_missing.index.intersection(numeric_cols)
insig_categorical = insig_missing.index.intersection(object_cols)

print(f"Insignificant missing values:\n{insig_missing}")

Insignificant missing values:
GarageType      81
GarageYrBlt     81
GarageFinish    81
GarageQual      81
GarageCond      81
BsmtExposure    38
BsmtFinType2    38
BsmtQual        37
BsmtCond        37
BsmtFinType1    37
MasVnrArea       8
Electrical       1
dtype: int64


## Insignificant Missing

There is a noticable pattern amongst the "Garage" related columns. They all all have a sum of 81 missing cells. 

In [5]:
no_garage = df[df["GarageQual"].isnull()][["GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond", "GarageArea"]]

print(no_garage)


     GarageType  GarageYrBlt GarageFinish GarageQual GarageCond  GarageArea
39          NaN          NaN          NaN        NaN        NaN           0
48          NaN          NaN          NaN        NaN        NaN           0
78          NaN          NaN          NaN        NaN        NaN           0
88          NaN          NaN          NaN        NaN        NaN           0
89          NaN          NaN          NaN        NaN        NaN           0
...         ...          ...          ...        ...        ...         ...
1349        NaN          NaN          NaN        NaN        NaN           0
1407        NaN          NaN          NaN        NaN        NaN           0
1449        NaN          NaN          NaN        NaN        NaN           0
1450        NaN          NaN          NaN        NaN        NaN           0
1453        NaN          NaN          NaN        NaN        NaN           0

[81 rows x 6 columns]


`GarageArea` was not part of the columns with missing cells, but I added it to show that the Garage related missing values aren't missing for no reason. Their respective garage areas (0) implies that the properties do not have a garage. It will be irresponsible of me to just fill it with with the mean or mode, because I can't make account of what does not exist; even if it's just a few rows. Instead, I will fill the alphabetic columns with "None", and the numeric ones with 0.

In [6]:
garage_cols_numeric = ["GarageYrBlt"]
garage_cols_categorical = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]

df[garage_cols_numeric] = df[garage_cols_numeric].fillna(0)
df[garage_cols_categorical] = df[garage_cols_categorical].fillna("None")

new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

BsmtExposure    38
BsmtFinType2    38
BsmtQual        37
BsmtCond        37
BsmtFinType1    37
MasVnrArea       8
Electrical       1
dtype: int64

`BsmtExposure`, `BsmtFinType2`, `BsmtQual`, `BsmtCond`, `BsmtFinType1` are all Basement related; similar to the Garage situation, but with a nuance. `BsmtExposure` and `BsmtFinType2` have 38 missing cells, while `BsmtQual`, `BsmtCond` and  `BsmtFinType1` have 37; 1 difference. Why? Let's investigate...


- First of all, I want to make sure that the 37's are on the same rows.

In [7]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
exposure_missing = set(df[df["BsmtExposure"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
fintype2_missing = set(df[df["BsmtFinType2"].isnull()].index)

qual_missing == cond_missing == fintype1_missing

True

It returns "True", so we're good.
- Secondly, I want to check if the 38's are supersets of the 37's

In [8]:
exposure_missing.issuperset(qual_missing) & fintype2_missing.issuperset(qual_missing)

True

It returns true, so we're good again
- Thirdly, I want to confirm that the two 38's are on the same row

In [9]:
exposure_missing == fintype2_missing

False

That's a problem there. Let's look further into the respective rows of the 38's that doesn't align

In [10]:
exposure_missing.symmetric_difference(fintype2_missing)

{332, 948}

Notice how the 38's are both supersets of the 37's, but their 1 row difference is casted into two seperate rows. `{332, 948}`

In [11]:
df.loc[[332, 948], ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "TotalBsmtSF"]]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,TotalBsmtSF
332,Gd,TA,No,GLQ,NaN,3206
948,Gd,TA,NaN,Unf,Unf,936


I have two decisions to make here; I drop the rows, or I fill it with the mode of the coluumns.
I will go with the latter.

In [12]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
no_basement = set(df[df["TotalBsmtSF"] == 0].index)

qual_missing == cond_missing == fintype1_missing == no_basement

True

The remaining basement-related NaN values have been confirmed to correspond to houses without a basement.
But a new question just arose; are the Basement related NAN cols the only Basement columns? Lets find out

In [13]:
[col for col in df.columns if "Bsmt" in col]

['BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinSF1',
 'BsmtFinType2',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 'BsmtFullBath',
 'BsmtHalfBath']

There are other seperate Basement related columns that were not listed as NAN. Let see what they have in them.

In [14]:
bsmt_categorical = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
bsmt_numeric_complete = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]

no_basement_rows = df[df[bsmt_categorical].isnull().any(axis=1)]

no_basement_rows[bsmt_numeric_complete].isnull().sum()

BsmtFinSF1      0
BsmtFinSF2      0
BsmtUnfSF       0
TotalBsmtSF     0
BsmtFullBath    0
BsmtHalfBath    0
dtype: int64

Good news, they are all numeric, and not null.
Finally, I will fill the Basement related NAN cells with `"None"`

In [15]:
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

df.fillna({col: "None" for col in bsmt_cols}, inplace=True)
df[bsmt_cols].isnull().sum() 

BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
dtype: int64

In [16]:
new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

MasVnrArea    8
Electrical    1
dtype: int64

The Basement related NAN cells have been filled. Time to look at the remaining two, starting with MasVnrArea.

In [17]:
[col for col in df.columns if "MasVnr" in col]

['MasVnrType', 'MasVnrArea']

I just checked for all the "Mas" related columns, and both happen to have missing cells, but with a nuance; `MasVnrType` has significantly more missing cells than `MasVnrArea`. Next, I'll check if `MasVnrArea` is a subset of `MasVnrType`.

In [18]:
type_missing = set(df[df["MasVnrType"].isnull()].index)
area_missing = set(df[df["MasVnrArea"].isnull()].index)

area_missing.issubset(type_missing)

True

Good news, `MasVnrArea` is a subset of `MasVnrType`.<br>
Next, I'll check for rows where `MasVnrType` is empty, but `MasVnrArea` isn't, and then check the value of `MasVnrArea` that falls within those columns. I'm doing this with the hope that every `MasVnrArea` cell that falls within those rows has a value of 0, because that would mean that every row where `MasVnrType` is NAN, the area is automatically 0.

In [19]:
mismatch = df[df["MasVnrType"].isnull() & df["MasVnrArea"].notna()]
mismatch["MasVnrArea"].value_counts()

MasVnrArea
0.0      859
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64

Well, that's a shame... Let' take a look at the none 0 rows

In [20]:
mismatch.loc[mismatch["MasVnrArea"] > 0, ["MasVnrType", "MasVnrArea"]]

,MasVnrType,MasVnrArea
624,NaN,288.0
773,NaN,1.0
1230,NaN,1.0
1300,NaN,344.0
1334,NaN,312.0


I'll fill these cells in `MasVnrType` with the mode of the column.

In [21]:
df.loc[[624, 773, 1230, 1300, 1334], "MasVnrType"] = df["MasVnrType"].mode()[0]

df.loc[[624, 773, 1230, 1300, 1334], ["MasVnrType", "MasVnrArea"]] # Confirm that the values have been updated correctly

,MasVnrType,MasVnrArea
624,BrkFace,288.0
773,BrkFace,1.0
1230,BrkFace,1.0
1300,BrkFace,344.0
1334,BrkFace,312.0


Now, I fill the missing `MasVnrArea` cells

In [22]:
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

df["MasVnrArea"].isna().sum()

np.int64(0)

Now I'll address the remaining missing values in `MasVnrType` by filling them with `"None"`. Before doing that, I'll confirm two things: that all rows with a missing `MasVnrType` have `MasVnrArea = 0`, and that the number of such rows matches the total count of missing cells in `MasVnrType` — ensuring there are no leftover inconsistencies before the fill.

In [23]:
mismatch = df[df["MasVnrType"].isnull() & df["MasVnrArea"].notna()]
mismatch["MasVnrArea"].value_counts()

MasVnrArea
0.0    867
Name: count, dtype: int64

In [24]:
df["MasVnrType"].isnull().sum()

np.int64(867)

In [25]:
area_zero_rows = set(df[df["MasVnrArea"] == 0].index)
type_null_rows = set(df[df["MasVnrType"].isnull()].index)

area_zero_rows == type_null_rows


False

The two groups (MasVnrArea == 0 and MasVnrType null) returned False when compared directly, meaning they aren't a perfect match. Before digging into which rows differ, I'll first check whether the two groups are even close in size — comparing their lengths will tell me if this is a small handful of edge cases or a larger, more concerning mismatch.

In [26]:
len(area_zero_rows), len(type_null_rows)

(869, 867)

 I also want to confirm that every row with a null `MasVnrType` has a corresponding `MasVnrArea` of 0 — in other words, that `type_null_rows` is a subset of `area_zero_rows`. If true, this would mean every "no type" row is also correctly reflected as "no area," even if the reverse isn't always the case.

In [27]:
type_null_rows.issubset(area_zero_rows)

True

Now that I know the two groups don't match exactly, I want to find the specific rows causing the mismatch. I'll take the difference between the two sets to isolate exactly which rows fall into one group but not the other, so I can inspect them directly and understand why they don't line up.

In [28]:
extra_rows = area_zero_rows - type_null_rows
df.loc[list(extra_rows), ["MasVnrType", "MasVnrArea"]]

,MasVnrType,MasVnrArea
688,BrkFace,0.0
1241,Stone,0.0


> Two rows (688 and 1241) have a valid `MasVnrType` (`BrkFace` and `Stone`) but a `MasVnrArea` of 0 — a real veneer type recorded with no corresponding area, which doesn't quite make physical sense. Rather than trusting the area over the type here, I'm treating these as inconsistent entries and overwriting `MasVnrType` to `"None"`, keeping the treatment consistent with the rest of the "no veneer" rows. This does mean discarding the originally recorded type for these two houses, but with only 2 rows affected, the impact on the overall analysis should be minimal.

**Verdict:** I'm treating `MasVnrType` as the more trustworthy field for these two rows, and fixing `MasVnrArea` instead of wiping the type. My reasoning: recording a veneer *type* is a quick visual judgment, while measuring the *area* takes actual effort — so a `0` area alongside a real type more likely reflects a lazy or incomplete measurement than a genuine absence of veneer. I'm filling each row's area with the median `MasVnrArea` for houses sharing the same veneer type (excluding other zero-area rows from that calculation), rather than using the dataset-wide mean or median, since area likely varies meaningfully by veneer type.

In [29]:
brkface_median = df[(df["MasVnrType"] == "BrkFace") & (df["MasVnrArea"] > 0)]["MasVnrArea"].median()
stone_median = df[(df["MasVnrType"] == "Stone") & (df["MasVnrArea"] > 0)]["MasVnrArea"].median()

df.loc[688, "MasVnrArea"] = brkface_median
df.loc[1241, "MasVnrArea"] = stone_median

In [30]:
area_zero_rows = set(df[df["MasVnrArea"] == 0].index)
type_null_rows = set(df[df["MasVnrType"].isnull()].index)

print(area_zero_rows == type_null_rows)
len(area_zero_rows), len(type_null_rows)

True


(867, 867)

Now they match perfectly — `area_zero_rows` and `type_null_rows` are exactly the same set of rows, confirming the `MasVnrType`/`MasVnrArea` inconsistencies have all been resolved. Time too fill the `MasVnrType`.

In [31]:
df["MasVnrType"] = df["MasVnrType"].fillna("None")

df["MasVnrType"].isnull().sum() # Should be 0 after filling missing values with "None"

np.int64(0)

The `MasVnr`-related columns have been fully cleaned — all inconsistencies between `MasVnrType` and `MasVnrArea` have been resolved, and no missing values remain in either column.

In [32]:
new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

Electrical    1
dtype: int64

Electrical — only 1 missing value, and unlike the Garage/Basement/MasVnr columns, there's no reasonable "doesn't apply" interpretation here (every house has an electrical system). This is almost certainly a genuine data-entry gap, not a meaningful NaN — filling with the mode is the appropriate, low-risk fix given it's a single row.

In [33]:
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [34]:
new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

Series([], dtype: int64)

**Congrats — `insig_missing` has been fully filled**, with a bonus of also fixing one column (`MasVnrType`/`MasVnrArea`) from the `sig_missing` group along the way. All that remains now are the truly significant missing-value columns.

## Significant Missing

In [35]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
FireplaceQu     690
LotFrontage     259
dtype: int64

Let's start from `PoolQC`

PoolArea: Pool area in square feet

PoolQC: Pool quality
		
       Ex	Excellent
       Gd	Good
       TA	Average/Typical
       Fa	Fair
       NA	No Pool

Here, there are two pool related columns. My hypothesis on this is that wherever `PoolQC` is NAN, the `PoolArea` is 0. Let's test this.

In [36]:
poolQ = set(df[df["PoolQC"].isnull()].index)
poolA = set(df[df["PoolArea"] == 0].index)

poolQ == poolA

True

Confirmed — PoolQC's missing rows match exactly with rows where PoolArea == 0, meaning every missing PoolQC value genuinely represents "no pool," not a data gap. Safe to fill with "None".

In [37]:
df["PoolQC"] = df["PoolQC"].fillna("None")

missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

MiscFeature    1406
Alley          1369
Fence          1179
FireplaceQu     690
LotFrontage     259
dtype: int64

`PoolQC` has been sorted. Next is `MiscFeature`

MiscFeature: Miscellaneous feature not covered in other categories
		
       Elev	Elevator
       Gar2	2nd Garage (if not described in garage section)
       Othr	Other
       Shed	Shed (over 100 SF)
       TenC	Tennis Court
       NA	None
		
MiscVal: $Value of miscellaneous feature

Same appraoch as `PoolQC`

In [38]:
Misc_feat = set(df[df["MiscFeature"].isnull()].index)
Misc_val = set(df[df["MiscVal"] == 0].index)

Misc_feat==Misc_val

False

That's strange. Let's look for the columns where they differ.

In [39]:
Misc_feat.symmetric_difference(Misc_val)

{873, 1200}

In [40]:
df.loc[[873, 1200], ["MiscFeature", "MiscVal"]]

,MiscFeature,MiscVal
873,Othr,0
1200,Shed,0


> Same situation as `MasVnrType`/`MasVnrArea` — these two rows have a real, recorded `MiscFeature` (`Othr` and `Shed`) but a `MiscVal` of 0. Following the same reasoning as before, recording *what* the feature is takes far less effort than actually appraising its dollar value, so a 0 value alongside a real feature type more likely reflects incomplete data entry than a genuinely worthless feature. Rather than overwriting `MiscFeature`, I'll fix `MiscVal` for these two rows using the median value for their respective feature type (excluding other zero-value rows from that calculation).

In [41]:
othr_median = df[(df["MiscFeature"] == "Othr") & (df["MiscVal"] > 0)]["MiscVal"].median()
shed_median = df[(df["MiscFeature"] == "Shed") & (df["MiscVal"] > 0)]["MiscVal"].median()

df.loc[873, "MiscVal"] = othr_median
df.loc[1200, "MiscVal"] = shed_median

Now, I'll compare them again

In [42]:
Misc_feat = set(df[df["MiscFeature"].isnull()].index)
Misc_val = set(df[df["MiscVal"] == 0].index)

print(Misc_feat==Misc_val)
df.loc[[873, 1200], ["MiscFeature", "MiscVal"]]

True


,MiscFeature,MiscVal
873,Othr,3500
1200,Shed,500


Perfect! Now they match and the cells has been corrected to the median value. Time to fill the NAN cells in `MiscFeature` with None.

In [43]:
df["MiscFeature"] = df["MiscFeature"].fillna("None")

df["MiscFeature"].isnull().sum() # Should be 0 after filling missing values with "None"

np.int64(0)

In [44]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

Alley          1369
Fence          1179
FireplaceQu     690
LotFrontage     259
dtype: int64

`MiscFeature`/`MiscVal` are now fully clean and consistent. Next up: `Alley`.

Moving on to Alley. Per the data description, NA here is a valid category meaning "no alley access," not a genuine gap — matching the ~94% missing rate found earlier. Before filling, I'll check whether alley access relates to SalePrice at all, given how rare it is.

In [45]:
df.groupby("Alley")["SalePrice"].mean()

Alley
Grvl    122219.080000
Pave    168000.585366
Name: SalePrice, dtype: float64

Alley is confirmed to represent a real, meaningful category rather than missing data — no numeric counterpart to cross-check against here (unlike Garage/Pool/Bsmt), but the data description directly confirms NA means "no alley access." Filling with "None" accordingly.

In [46]:
df["Alley"] = df["Alley"].fillna("None")

df["Alley"].isnull().sum() # Should be 0 after filling missing values with "None"

np.int64(0)

In [47]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

Fence          1179
FireplaceQu     690
LotFrontage     259
dtype: int64

Next up: `Fence`. Same approach as `Alley` — checking the data description first to confirm what `NA` actually represents before deciding how to handle the missing values.

In [48]:
df.groupby("Fence")["SalePrice"].mean()

Fence
GdPrv    178927.457627
GdWo     140379.314815
MnPrv    148751.089172
MnWw     134286.363636
Name: SalePrice, dtype: float64

The spread across fence types isn't dramatic, but I'll keep the column rather than drop it. Once I fill the missing values with `"None"`, I'll re-check the group averages — the "no fence" category is by far the largest group, so seeing where it lands relative to the fenced types should give a more complete picture.

In [49]:
df["Fence"] = df["Fence"].fillna("None")
df.groupby("Fence")["SalePrice"].mean()

Fence
GdPrv    178927.457627
GdWo     140379.314815
MnPrv    148751.089172
MnWw     134286.363636
None     187596.837998
Name: SalePrice, dtype: float64

Counterintuitive finding: houses with **no fence** actually have the *highest* average `SalePrice` (187,597) — higher than every fenced category. My initial assumption was that no fence would mean a cheaper, less-maintained property, but the opposite is true.
>
> Likely explanation: fences tend to appear on smaller, more private lots where owners want boundary/privacy control, while larger or more spacious properties often don't need one at all. `Fence` may be acting as a proxy for lot size or property type rather than directly reflecting home value or quality — worth keeping in mind rather than treating fence presence as a straightforward price signal.
>
Let's find out if fence is really a proxt for LotArea.

In [50]:
df.groupby("Fence")["LotArea"].mean()

Fence
GdPrv    10520.288136
GdWo      9634.018519
MnPrv     9204.458599
MnWw      9250.909091
None     10743.659881
Name: LotArea, dtype: float64

No-fence properties do have the largest average `LotArea`, and `GdPrv` is close behind — mildly consistent with the lot-size theory. But the spread here (roughly 1,000–1,500 sq ft) is far too small to fully explain the ~$50,000 gap in average `SalePrice` between `None` and the cheapest fence category (`MnWw`). **Verdict: lot size is likely part of the explanation, but not the whole story** — something else (possibly overall house quality or neighborhood) is probably also at play. Leaving this as an open, partially-explained finding rather than forcing a conclusion the data doesn't fully support.

In [51]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

FireplaceQu    690
LotFrontage    259
dtype: int64

> Next up: `FireplaceQu` (Fireplace Quality). Same approach — check the data description for what `NA` means, then test the relationship with `SalePrice` before filling.
>
>FireplaceQu has a numeric counterpart — Fireplaces (count of fireplaces on the property). Same verification pattern as Garage/Pool: if FireplaceQu is missing, Fireplaces should be exactly 0, confirming "no fireplace" rather than a genuine data gap.

In [52]:
fireqc_missing = set(df[df["FireplaceQu"].isnull()].index)
fireplaces_zero = set(df[df["Fireplaces"] == 0].index)

fireqc_missing == fireplaces_zero

True

In [53]:
df["FireplaceQu"] = df["FireplaceQu"].fillna("None")

In [54]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

LotFrontage    259
dtype: int64

Next up: LotFrontage (259 missing, ~18%). Unlike Garage/Pool/Bsmt, there's no reason to believe missing frontage means "no frontage" — every house has a lot with some width facing the street, so this is likely genuine missing data rather than a "doesn't apply" case. A flat mean/median fill is the default option, but before committing to that, I want to check if there's a better, more informed way to estimate it.

In [55]:
df["LotFrontage"].corr(df["SalePrice"])

np.float64(0.351799096570678)

LotFrontage correlates moderately with SalePrice (0.351) — a real relationship, so this column is worth keeping and filling carefully rather than dropping. Checking whether LotArea could help predict missing frontage values, since both describe the size/shape of the lot.

In [56]:
df["LotFrontage"].corr(df["LotArea"])

np.float64(0.42609501877180805)

LotFrontage and LotArea show a moderate correlation (0.42) — some relationship, but not strong enough to rely on alone. Hypothesis: houses in the same Neighborhood likely share similar lot layouts (consistent subdivision patterns, similar development eras), so neighborhood might be a better predictor of frontage than lot area alone. Testing this before deciding on a fill strategy.

In [57]:
df.groupby("Neighborhood")["LotFrontage"].median()

Neighborhood
Blmngtn    43.0
Blueste    24.0
BrDale     21.0
BrkSide    52.0
ClearCr    80.0
CollgCr    70.0
Crawfor    74.0
Edwards    65.5
Gilbert    65.0
IDOTRR     60.0
MeadowV    21.0
Mitchel    73.0
NAmes      73.0
NPkVill    24.0
NWAmes     80.0
NoRidge    91.0
NridgHt    88.5
OldTown    60.0
SWISU      60.0
Sawyer     71.0
SawyerW    66.5
Somerst    73.5
StoneBr    61.5
Timber     85.0
Veenker    68.0
Name: LotFrontage, dtype: float64

The neighborhood medians show real, meaningful spread — from 21.0 (Blueste, BrDale, MeadowV) up to 91.0 (NoRidge), over a 4x difference. To properly test whether this would make a good fill strategy, checking how well the neighborhood median would have predicted each house's actual known LotFrontage, using .transform() to map each row to its own neighborhood's median without collapsing the DataFrame.

In [58]:
df["LotFrontage"].corr(df.groupby("Neighborhood")["LotFrontage"].transform("median"))

np.float64(0.43941317243019495)

The neighborhood-median approach correlates with the real LotFrontage values at 0.439 — slightly better than using LotArea alone (0.42), and clearly more informed than a single flat mean/median across the whole dataset. Verdict: filling missing LotFrontage values using each house's own neighborhood median, rather than a flat overall value.

In [59]:
df["LotFrontage"] = df["LotFrontage"].fillna(df.groupby("Neighborhood")["LotFrontage"].transform("median"))
df["LotFrontage"].isnull().sum()

np.int64(0)

In [60]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0].sort_values(ascending=False)
missing_now

Series([], dtype: int64)

> All missing values across the dataset have now been filled. Each fill was informed by an actual investigation into *why* the value was missing, rather than a blanket mean/mode/None applied indiscriminately. Several columns required catching and individually correcting genuine data-entry inconsistencies hiding within otherwise clean "doesn't apply" patterns (Basement, MasVnr, Misc), and `LotFrontage` was filled using a neighborhood-based median, validated against the real data before committing to it.
>
> Missing-value cleaning is complete. Next: moving into univariate analysis.

**Checkpoint: saving the cleaned dataset.** All missing-value handling is complete and verified — saving to train_cleaned.csv as a stable checkpoint before moving into univariate analysis. This means the notebook can be re-run from this point forward without repeating the full cleaning investigation, and provides a clear before/after artifact of the cleaning process for the repo.

In [61]:
df.isnull().sum().sum()

np.int64(0)

In [62]:
df.to_csv("../data/train_cleaned.csv", index=False)

In [63]:
df.isnull().sum().sum()

np.int64(0)